In [2]:
import torch
import torch.nn as nn

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)

from torch.utils.data import DataLoader

# =====================================
# CONFIG
# =====================================

MAX_LEN = 512

BATCH_SIZE = 16

EPOCHS = 5

LR = 2e-5

MODEL_NAME = "bert-base-uncased"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE:", DEVICE)

# =====================================
# LOAD DATASET
# =====================================

dataset = load_dataset("ag_news")

train_dataset = dataset["train"]

test_dataset = dataset["test"]

NUM_CLASSES = 4

# =====================================
# TOKENIZER
# =====================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

def tokenize(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True
)

train_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

test_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

# =====================================
# DATALOADER
# =====================================

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

DEVICE: cuda


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [3]:
# =====================================
# MODEL
# =====================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES
)

model.to(DEVICE)

# =====================================
# OPTIMIZER
# =====================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR
)

criterion = nn.CrossEntropyLoss()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
import time
import torch

from tqdm import tqdm

# =====================================
# CUDA OPTIMIZATION
# =====================================

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

# =====================================
# COMPILE MODEL
# =====================================

# tăng speed đáng kể trên Kaggle
model = torch.compile(model)

# =====================================
# MIXED PRECISION
# =====================================

scaler = torch.amp.GradScaler('cuda')

# =====================================
# TRAIN
# =====================================

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    correct = 0
    total = 0

    start = time.time()

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    for batch in tqdm(train_loader):

        # =========================
        # LOAD GPU
        # =========================

        input_ids = batch["input_ids"].to(
            DEVICE,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            DEVICE,
            non_blocking=True
        )

        labels = batch["label"].to(
            DEVICE,
            non_blocking=True
        )

        # =========================
        # ZERO GRAD
        # =========================

        optimizer.zero_grad(
            set_to_none=True
        )

        # =========================
        # MIXED PRECISION
        # =========================

        with torch.amp.autocast(
            device_type='cuda',
            dtype=torch.float16
        ):

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits

            loss = criterion(
                logits,
                labels
            )

        # =========================
        # BACKWARD
        # =========================

        scaler.scale(loss).backward()

        # bỏ clipping để tăng speed
        # torch.nn.utils.clip_grad_norm_(
        #     model.parameters(),
        #     1.0
        # )

        scaler.step(optimizer)

        scaler.update()

        # =========================
        # METRICS
        # =========================

        total_loss += loss.item()

        preds = torch.argmax(
            logits,
            dim=1
        )

        correct += (preds == labels).sum().item()

        total += labels.size(0)

    # =====================================
    # RESULT
    # =====================================

    end = time.time()

    acc = correct / total

    avg_loss = total_loss / len(train_loader)

    print(f"\nLoss: {avg_loss:.4f}")

    print(f"Accuracy: {acc:.4f}")

    print(f"Time: {end-start:.2f} sec")


Epoch 1/5


100%|██████████| 7500/7500 [47:50<00:00,  2.61it/s]



Loss: 0.2097
Accuracy: 0.9286
Time: 2870.39 sec

Epoch 2/5


100%|██████████| 7500/7500 [47:10<00:00,  2.65it/s]



Loss: 0.1274
Accuracy: 0.9564
Time: 2830.02 sec

Epoch 3/5


...


Loss: 0.0873
Accuracy: 0.9697
Time: 2833.44 sec

Epoch 4/5


100%|██████████| 7500/7500 [47:11<00:00,  2.65it/s]



Loss: 0.0600
Accuracy: 0.9793
Time: 2831.34 sec

Epoch 5/5


100%|██████████| 7500/7500 [47:10<00:00,  2.65it/s]


Loss: 0.0431
Accuracy: 0.9851
Time: 2830.73 sec
